In [ ]:
import os
import glob
import numpy as np
from tqdm import tqdm
from PIL import Image
import pydicom
from sklearn.pipeline import Pipeline
from gtda.homology import CubicalPersistence
from gtda.diagrams import PersistenceImage
import matplotlib.pyplot as plt

DATA_DIR = r"C:\Users\omer_\Documents\GitHub\ytustats\standard_u-net\Brain_Stroke_CT_Dataset"
IMAGE_TYPE_FOLDER = 'DICOM' 
FILE_EXTENSION = '.dcm'
# Karışmaması için yeni ve temiz bir klasör adı
TDA_OUTPUT_DIR = "./topological_features" 
os.makedirs(TDA_OUTPUT_DIR, exist_ok=True)


#PIPELINE
cubical_persistence = CubicalPersistence(homology_dimensions=(0, 1), n_jobs=-1)
persistence_image = PersistenceImage(n_bins=32, n_jobs=-1)
tda_pipeline = Pipeline([
    ("cubical_persistence", cubical_persistence),
    ("persistence_image", persistence_image)
])
print("Giotto-tda pipeline'ı (v0.6.2 - CubicalPersistence) ile başarıyla tasarlandı.")

In [ ]:
all_image_files = []
for folder in ["Bleeding", "Ischemia", "Normal"]:
    class_dir = os.path.join(DATA_DIR, folder, IMAGE_TYPE_FOLDER)
    files = glob.glob(os.path.join(class_dir, f'*{FILE_EXTENSION}'))
    all_image_files.extend(files)
external_test_dir = os.path.join(DATA_DIR, 'External_Test', IMAGE_TYPE_FOLDER)
if os.path.isdir(external_test_dir):
    all_image_files.extend(glob.glob(os.path.join(external_test_dir, f'*{FILE_EXTENSION}')))
print(f"\nİşlenmek üzere toplam {len(all_image_files)} adet görüntü dosyası bulundu.")

In [ ]:
print("\nTopolojik imzalar üretiliyor ve .npy olarak kaydediliyor...")
for file_path in tqdm(all_image_files, desc="Topolojik İmza Üretimi"):
    try:
        # --- Görüntüyü Oku ve 2D Hale Getir ---
        if FILE_EXTENSION == '.dcm':
            image_array = pydicom.dcmread(file_path).pixel_array
        else:
            image_array = np.array(Image.open(file_path).convert("L"))

        if image_array.ndim == 3:
            middle_slice_index = image_array.shape[0] // 2
            final_image_2d = image_array[middle_slice_index, :, :]
        elif image_array.ndim == 2:
            final_image_2d = image_array
        else:
            continue
        # CubicalPersistence, (n_samples, height, width) şeklinde 3D bir dizi bekler.
        # Nokta bulutu, yeniden boyutlandırma, sahte RGB gibi adımlara GEREK YOKTUR.
        image_for_gtda = final_image_2d[np.newaxis, :, :]
        
        persistence_images = tda_pipeline.fit_transform(image_for_gtda)
        topological_feature = persistence_images[0]
        
        base_filename = os.path.basename(file_path)
        filename_without_ext = os.path.splitext(base_filename)[0]
        output_npy_path = os.path.join(TDA_OUTPUT_DIR, f"{filename_without_ext}.npy")
        np.save(output_npy_path, topological_feature)

    except Exception as e:
        print(f"\nHATA: '{file_path}' dosyası işlenirken bir sorun oluştu.")
        print(f"Hata mesajı: {e}")
        continue

print("\nİşlem tamamlandı!")
print(f"Tüm topolojik imzalar '{TDA_OUTPUT_DIR}' klasörüne kaydedildi.")